# **Практика 4. Генерация контента с помощью GigaChat API**

**Цель:** научиться работать с российским LLM API прямо из Python-кода.

## Почему GigaChat

- Разработан Сбером, работает в России без VPN
- Бесплатный доступ через личный кабинет Сбера
- Хорошо понимает русский язык и деловой контекст
- Есть официальная Python-библиотека

## Как получить доступ

1. Зайдите на https://developers.sber.ru/studio/
2. Войдите через СберID (нужен аккаунт Сбера)
3. Создайте проект → получите Client ID и Client Secret
4. Вставьте их в переменные ниже

> **Альтернатива:** Если не получилось с GigaChat — используйте Gemini API через Google AI Studio (aistudio.google.com) — там тоже бесплатно и без VPN. Также можно воспользоваться сервисом OpenRouter.

OpenRouter: https://openrouter.ai/

In [ ]:
# Устанавливаем библиотеки
!pip install gigachat google-generativeai -q
print("Готово!")

In [ ]:
# ====================================================
# НАСТРОЙКА: выберите один вариант API
# ====================================================

# Вариант A: GigaChat (Сбер)
GIGACHAT_CREDENTIALS = ""  # из личного кабинета

Если выбираете не ГигаЧат, а другой сервис, не забывайте поменять функцию ниже и скачать нужные библиотеки.

In [ ]:
def ask_llm(prompt, system="Ты профессиональный PR-менеджер.", temperature=0.7):
    """Отправляет запрос к LLM и возвращает ответ."""
    from gigachat import GigaChat
    from gigachat.models import Chat, Messages, MessagesRole

    with GigaChat(credentials=GIGACHAT_CREDENTIALS, verify_ssl_certs=False) as giga:
        response = giga.chat(Chat(
            model="GigaChat-2", # если у вас есть оплаченные токены, модель можно поменять
            messages=[
                Messages(role=MessagesRole.SYSTEM, content=system),
                Messages(role=MessagesRole.USER, content=prompt)
            ],
            temperature=temperature
        ))
    return response.choices[0].message.content

In [ ]:
print("Тестируем подключение...")

result = ask_llm("Скажи 'Привет от API' и больше ничего.")

print(f"Ответ: {result}")
print("API работает!")

---
## Задание 1: Суммаризация статей

In [ ]:
def summarize_article(article_text, max_sentences=3):

    """Создаёт краткое резюме статьи."""

    prompt = f"""Сделай краткое резюме этой новости в {max_sentences} предложениях.
                 Пиши лаконично, только факты. Без вводных слов типа "В статье говорится".
                 Новость:
                 {article_text[:2000]}"""

    return ask_llm(prompt, system="Ты редактор новостного агентства. Пишешь кратко и по существу.")

In [ ]:
# Тест на примере
sample_news = """
Яндекс объявил о запуске новой версии голосового помощника Алиса,
которая получила улучшенную модель понимания естественного языка и поддержку
многоходовых диалогов. Обновление затрагивает все устройства Яндекса, включая
смарт-колонки и приложение для смартфонов. Компания отмечает, что новая Алиса
лучше справляется со сложными запросами и помнит контекст разговора.
Тестирование проходило в течение полугода с участием 10 000 пользователей.
"""

summary = summarize_article(sample_news)
print("Оригинал (первые 200 символов):")
print(sample_news[:200] + "...")
print("\nРезюме:")
print(summary)

---
## Задание 2: Мультиканальный адаптер контента

In [ ]:
def adapt_for_channels(news_text, channels=None):

    """Адаптирует новость для разных каналов."""

    if channels is None:
        channels = ['vk', 'telegram', 'site', 'email']

    channel_specs = {
        'vk': 'VK-пост: неформальный, живой, до 300 символов, можно с эмодзи',
        'telegram': 'Telegram-пост: краткий, конкретный, до 200 символов, 1-2 эмодзи',
        'site': 'Текст для официального сайта: деловой стиль, 2-3 предложения, без эмодзи',
        'email': 'Вступление email-рассылки: приветливо, до 100 слов, с обращением «Уважаемые читатели»'
    }

    results = {}

    for channel in channels:
        if channel not in channel_specs:
            continue

        prompt = f"""Адаптируй эту новость для следующего канала:
        Канал: {channel_specs[channel]}

        Исходная новость:
        {news_text}

        Напиши ТОЛЬКО адаптированный текст, без пояснений."""

        results[channel] = ask_llm(prompt)

    return results

In [ ]:
# Пример
news = "Компания Greentech запустила приложение для отслеживания углеродного следа. " \
       "Пользователи могут видеть свой экологический след в реальном времени и получать советы по его снижению."

adaptations = adapt_for_channels(news)

for channel, text in adaptations.items():
    print(channel.upper())
    print(text)

---
## Задание 3: Генератор заголовков

In [ ]:
def generate_headlines(news_text, n=5):

    """Генерирует N вариантов заголовка для новости."""

    prompt = f"""Придумай {n} разных заголовков для этой новости.
    Каждый заголовок — в отдельной строке, без нумерации и пояснений.
    Стили: официальный, разговорный, интригующий, с цифрой, вопросительный.

    Новость:
    {news_text}"""

    response = ask_llm(prompt, temperature=0.9)
    headlines = [h.strip() for h in response.split('\n') if h.strip()]

    return headlines[:n]

In [ ]:
news = "Молодая компания из Казани разработала ИИ-систему, которая предсказывает вирусность контента " \
       "до его публикации с точностью 78%. Система уже используется тремя крупными медиахолдингами."

headlines = generate_headlines(news, n=5)
print("Варианты заголовков:")
for headline in headlines:
    print(headline)

# Автоматизация анализа отзывов и комментариев

**Цель:** создать анализатор тональности, который обрабатывает сотни отзывов и формирует отчёт.

**Что делаем:**
1. Анализ тональности отзывов через LLM
2. Автоматический отчёт для руководителя
3. NER — кто и что упоминается в отзывах

In [ ]:
import json

In [ ]:
def analyze_review_llm(review_text):

    """Анализирует один отзыв через LLM, возвращая JSON."""

    prompt = f"""Проанализируй этот отзыв. Верни ТОЛЬКО JSON без пояснений:
    {{
      "sentiment": "positive" или "negative" или "mixed",
      "score": число от -1.0 (очень негативный) до 1.0 (очень позитивный),
      "topic": главная тема отзыва одним словом (доставка/качество/сервис/цена/приложение/другое),
      "key_issue": главная проблема или похвала в одном предложении
    }}

    Отзыв: {review_text}"""

    response = ask_llm(prompt, temperature=0.1)

    try:
        clean = response.strip()
        if clean.startswith('```'):
            clean = clean.split('```')[1]
            if clean.startswith('json'):
                clean = clean[4:]
        return json.loads(clean)
    except json.JSONDecodeError:
        # Fallback for LLM responses that are not valid JSON
        return {"sentiment": "neutral", "score": 0, "topic": "другое", "key_issue": response[:100]}

In [ ]:
# Пример использования:
sample_review = "Приложение очень удобное, но иногда медленно загружается."
analysis_result = analyze_review_llm(sample_review)
print(analysis_result)

---
## Датасет: отзывы на приложение банка
Можно скачать по ссылке: https://raw.githubusercontent.com/dianasusla28/andan_2026/refs/heads/main/Bank_reviews.csv

In [ ]:
import pandas as pd
reviews = pd.read_csv('https://raw.githubusercontent.com/dianasusla28/andan_2026/refs/heads/main/Bank_reviews.csv')
reviews

---
## Шаг 1: Анализ тональности каждого отзыва

In [ ]:
import time

# Анализируем первые 5 отзывов для демонстрации
analyzed = []
counter = 1

for review in reviews.values[:5]:
    review_text = review[0]

    print(f"Анализирую отзыв {counter}/5...")

    result = analyze_review_llm(review_text)

    result['review_text'] = review_text

    analyzed.append(result)

    counter += 1

# Выводим результаты
print("\nРезультаты:")

for item in analyzed:

    if item['sentiment'] == 'positive':
        emoji = '✅'
    elif item['sentiment'] == 'negative':
        emoji = '❌'
    elif item['sentiment'] == 'mixed':
        emoji = '➖'
    else:
        emoji = '?'

    print(f"{emoji} [{item['topic']}] {item['review_text']}")
    print(f"   Оценка: {item['score']:.1f} | {item['key_issue']}")

In [ ]:
analyzed

---
## Шаг 2: Автоматический отчёт

In [ ]:
def generate_report(analyzed_reviews):

    """Генерирует текстовый отчёт для руководителя."""

    sentiments = [r['sentiment'] for r in analyzed_reviews]
    pos_count = sentiments.count('positive')
    neg_count = sentiments.count('negative')
    neu_count = sentiments.count('mixed')

    negative_issues = [r['key_issue'] for r in analyzed_reviews if r['sentiment'] == 'negative']
    positive_issues = [r['key_issue'] for r in analyzed_reviews if r['sentiment'] == 'positive']

    prompt = f"""На основе анализа {len(analyzed_reviews)} отзывов напиши короткий отчёт для PR-директора (150-200 слов) в формате email.
                  Тон уважительный, официальный.

                  Статистика:
                  - Позитивных: {pos_count}
                  - Негативных: {neg_count}
                  - Нейтральных: {neu_count}

                  Основные жалобы:
                  {chr(10).join(['- ' + i for i in negative_issues[:5]])}

                  Основные похвалы:
                  {chr(10).join(['- ' + i for i in positive_issues[:5]])}

                  Структура отчёта:
                  1. Общая оценка (1-2 предложения)
                  2. Главные проблемы (2-3 пункта)
                  3. Что нравится пользователям (2-3 пункта)
                  4. Рекомендации для PR-команды (2-3 пункта)"""

    return ask_llm(prompt, system="Ты аналитик данных в компании Сбер, пишешь для бизнес-аудитории.")

In [ ]:
report = generate_report(analyzed)
print("ОТЧЁТ ДЛЯ PR-ДИРЕКТОРА")
print("=" * 50)
print(report)

---
## Шаг 3: NER — кто и что упоминается в отзывах

**Зачем это нужно в PR?**

Анализ тональности говорит нам *как* пишут. NER говорит *о чём* и *о ком* именно.
Из отзывов на банковское приложение можно узнать:
- Какие конкуренты упоминаются рядом с вашим брендом?
- Какие функции и сервисы называют по имени?
- Какие персоны (сотрудники, публичные лица) появляются в отзывах?


###  NER через LLM-промпт


In [ ]:
def ner_with_llm(text):

    """Извлекает сущности через LLM."""

    prompt = f"""Найди в тексте все упоминания. Верни ТОЛЬКО JSON без пояснений:
                {{
                  "organizations": [список организаций, банков, сервисов, приложений],
                  "features": [список упомянутых функций или возможностей приложения],
                  "problems": [конкретные технические проблемы, если есть]
                }}

                Текст: {text}"""

    response = ask_llm(prompt, temperature=0.1)
    try:
        clean = response.strip()
        if clean.startswith('```'):
            clean = clean.split('```')[1]
            if clean.startswith('json'):
                clean = clean[4:]
        return json.loads(clean)
    except json.JSONDecodeError:
        return {"organizations": [], "features": [], "problems": []}

In [ ]:
# Применяем к первым 5 отзывам
print("NER через LLM — первые 5 отзывов:")
print("=" * 50)

all_features = []
all_problems_ner = []

for i, review_text in enumerate(reviews['Отзыв'][:5]):
    print(f"\nОтзыв {i+1}: {review_text}...")
    entities = ner_with_llm(review_text)

    if entities.get('organizations'):
        print(f"  Организации: {', '.join(entities['organizations'])}")
    if entities.get('features'):
        print(f"  Функции: {', '.join(entities['features'])}")
        all_features.extend(entities['features'])
    if entities.get('problems'):
        print(f"  Проблемы: {', '.join(entities['problems'])}")
        all_problems_ner.extend(entities['problems'])

### Объединяем тональность + NER

Самое полезное для PR — знать не просто «много негатива»,
а **что именно** вызывает негатив и **что** — позитив.

In [ ]:
# Объединяем тональность + NER через LLM
print("ФУНКЦИИ И ПРОБЛЕМЫ В ОТЗЫВАХ (через LLM)")
print("=" * 50)

negative_features = []
positive_features = []
negative_problems = []

for i, item in enumerate(analyzed):

    entities = ner_with_llm(item['review_text'])

    if item['sentiment'] == 'negative':
        negative_features.extend(entities.get('features', []))
        negative_problems.extend(entities.get('problems', []))
        print(f" {item['review_text'][:60]}...")
        if entities.get('problems'):
            print(f"     Проблемы: {', '.join(entities['problems'])}")
        if entities.get('features'):
            print(f"     Функции: {', '.join(entities['features'])}")

    elif item['sentiment'] == 'positive':
        positive_features.extend(entities.get('features', []))
        print(f"  {item['review_text'][:60]}...")
        if entities.get('features'):
            print(f"     Функции: {', '.join(entities['features'])}")

In [ ]:
from collections import Counter

In [ ]:
print("САМЫЕ ЧАСТЫЕ ПРОБЛЕМЫ В НЕГАТИВНЫХ ОТЗЫВАХ:")
for problem, count in Counter(negative_problems).most_common(10):
    print(f"  '{problem}': {count}")

print("\nФУНКЦИИ, УПОМЯНУТЫЕ В НЕГАТИВНЫХ ОТЗЫВАХ:")
for feature, count in Counter(negative_features).most_common(10):
    print(f"  '{feature}': {count}")

print("\nФУНКЦИИ, УПОМЯНУТЫЕ В ПОЗИТИВНЫХ ОТЗЫВАХ:")
for feature, count in Counter(positive_features).most_common(10):
    print(f"  '{feature}': {count}")

---
## Самостоятельное задание

1. Проанализируйте **все 30 отзывов** (замените `[:5]` на полный датасет)
2. Найдите все **негативные** отзывы и выведите их список
3. Для каждого негативного отзыва сгенерируйте **ответ компании** (2–3 предложения, вежливо, с предложением решения)
4. Запустите **NER** на всех отзывах и выведите топ-5 самых часто упоминаемых функций
5. Сохраните финальный отчёт в текстовый файл `report.txt`, добавив в конец раздел **«Что упоминают пользователи»** — список топ-организаций и топ-функций из NER

**Для получения балла** загрузите файл в форму.  

Если вы использовали ИИ, подробно опишите для чего (если не использовали, тоже укажите это) - **РАБОТЫ БЕЗ ДЕКЛАРАЦИИ ОБ ИСПОЛЬЗОВАНИИ ИИ ОЦЕНИВАЮТСЯ В 0 БАЛЛОВ!**

In [ ]:
reviews = list(reviews['Отзыв'])

In [ ]:
reviews

In [ ]:
def generate_negative_report(): # не забудьте передать параметры

  # ваш код здесь

  return report

Не забудьте сгенерировать txt-файл и прикрепить его в форму. Можно создавать дополнительные функции.

In [ ]:
# ваш код здесь

За что еще могут отниматься баллы:

1. Лишние вызовы LLM
2. Очевидно ИИшный код без редактирования
3. За все, за что обычно снимаются баллы
4. За скорпированную работу - 0 баллов
5. За код, который не запускается
6. Оставили ключ от LLM в коде
7. Перед сдачей стерли вывод ячеек